# 第11课 - 代理到代理 (A2A) 协议


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 什么是 A2A 协议？

**代理到代理 (A2A) 协议** 是一个开放标准，使 AI 代理能够相互通信、发现彼此并协作 —— 即使它们建立在不同的框架上或由不同的服务托管。

关键概念：

- **发现** – 代理发布一个 *代理卡*，描述其能力，使其他代理（或协调器）更容易为任务找到合适的专家。
- **消息传递** – 代理通过通用协议交换结构化消息，因此一个代理的请求可以被另一个代理理解并完成，而不管其内部实现如何。
- **任务生命周期** – A2A 定义了诸如 *已提交*、*处理中*、*已完成* 和 *失败* 等状态，使协调器能够完整地查看委派任务的进展。

在本课中，我们通过将三个专业的旅行代理连接到一个工作流中来模拟 A2A 风格的协作，每个代理贡献其专业知识并将结果传递给下一个。


## 创建专业化的旅行代理


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 通过工作流进行多代理协作

我们将这三个代理连接成一个顺序工作流，模拟 A2A 消息传递：

1. **CurrencyExchangeAgent** 接收用户请求并提供货币兑换建议。
2. **ActivityPlannerAgent** 接收丰富的上下文并添加活动推荐。
3. **TravelManagerAgent** 将两者的输入汇总为最终的旅行简报。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 在生产环境中理解 A2A

在生产环境中，A2A 协议解锁了强大的跨服务场景：

| 能力 | 描述 |
|---|---|
| **跨框架互操作** | 由一个框架构建的代理可以将任务委派给由任何其他与 A2A 兼容的框架构建的代理，从而实现真正的跨组织互操作性。 |
| **服务边界** | 代理可以部署在独立的微服务、云区域，甚至不同的组织中，同时仍能无缝协作。 |
| **动态发现** | 编排器可以在运行时查询 Agent Card 注册表以找到最适合某个子任务的专用代理。 |
| **流式传输与推送通知** | A2A 支持服务器发送事件 (SSE) 用于实时进度更新，并为长时间运行的任务提供推送通知。 |

我们上面构建的工作流是该模式的一个简化的进程内版本。在实际
部署中，每个代理都会暴露一个 HTTP 端点、发布一个 Agent Card，并进行通信
通过 A2A JSON-RPC 协议。


## 总结

在本课中你学到了：

1. **什么是 A2A 协议** — 一种用于代理间发现、消息传递，
   和任务管理。
2. **如何创建专用代理** — 一个货币兑换代理、一个活动规划代理，
   和一个旅行管理编排器。
3. **如何将代理连接到工作流** — 使用 `WorkflowBuilder` 来建模代理之间的顺序
   消息传递。
4. **A2A 在生产环境中的工作方式** — 实现跨框架、跨服务的协作
   通过动态发现和流式更新。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免责声明：
本文件使用人工智能翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文件应被视为权威来源。对于重要信息，建议采用专业人工翻译。因使用本翻译而产生的任何误解或曲解，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
